# AI4EAC Finance Practice Challenge
- To develop a robust, generalisable machine learning model to accurately predict the likelihood of loan defaults for both existing customers and new applicants. 
- Incorporate unique factors relevant to each financial market.

### Current models
- Baseline models: Random forest
- Best performing: XGboost,gradient boosting
- LightGBM:Fater,memory efficient
- Adaboost: Imbalanced datasets

### Data pipeline
1) Load the data
2) Explore and visualize the data
3) Data preprocessing and feature engineering
4) Train models
5) Select and finetune the best performing model
6) Evaluate

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
import warnings
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

N_FOLDS = 5
SEED    = 42

c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Load the data

In [2]:
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')

### 3. Data preprocessing and feature engineering


In [3]:
# disbursement_date and due_date are strings 

for df in [train, test]:
    for col in ['disbursement_date', 'due_date']:
        dt = pd.to_datetime(df[col])
        df[f'{col}_month']      = dt.dt.month
        df[f'{col}_dow']        = dt.dt.dayofweek
        df[f'{col}_quarter']    = dt.dt.quarter
        df[f'{col}_year']       = dt.dt.year
        df[f'{col}_is_weekend'] = (dt.dt.dayofweek >= 5).astype(int)
        df[f'{col}_day']        = dt.dt.day

    df.drop(columns=['disbursement_date', 'due_date'], inplace=True)

In [4]:
# 35% of loans have zero interest — strong signal on its own, Negative interest exists (loans repaid less than borrowed) , flag it

for df in [train, test]:
    df['interest_charged']     = df['Total_Amount_to_Repay'] - df['Total_Amount']
    df['interest_rate']        = df['interest_charged'] / (df['Total_Amount'] + 1)
    df['is_zero_interest']     = (df['interest_charged'] == 0).astype(int)
    df['is_negative_interest'] = (df['interest_charged'] < 0).astype(int)

In [5]:
# Lender_portion_Funded > 1 is a data anomaly - flag it

for df in [train, test]:
    df['lender_exposure_ratio']    = df['Amount_Funded_By_Lender'] / (df['Total_Amount'] + 1)
    df['lender_return_ratio']      = df['Lender_portion_to_be_repaid'] / (df['Amount_Funded_By_Lender'] + 1)
    df['lender_funded_over1_flag'] = (df['Lender_portion_Funded'] > 1).astype(int)
    df['lender_amount_per_day']    = df['Amount_Funded_By_Lender'] / (df['duration'] + 1)

In [6]:
# Total_Amount is heavily right-skewed (max=20M, median=5.2K)
# Log transform compresses the scale for tree models

for df in [train, test]:
    df['log_total_amount']    = np.log1p(df['Total_Amount'].clip(lower=0))
    df['log_amount_to_repay'] = np.log1p(df['Total_Amount_to_Repay'].clip(lower=0))
    df['amount_per_day']      = df['Total_Amount'] / (df['duration'] + 1)
    df['repay_per_day']       = df['Total_Amount_to_Repay'] / (df['duration'] + 1)

In [7]:
# 94.7% are 7-day loans — duration is almost a constant but non-7 is a signal

for df in [train, test]:
    df['is_7day_loan']   = (df['duration'] == 7).astype(int)
    df['is_14day_loan']  = (df['duration'] == 14).astype(int)
    df['is_30plus_loan'] = (df['duration'] >= 30).astype(int)
    df['log_duration']   = np.log1p(df['duration'])

In [ ]:
# customer history aggregation using expanding windows sort train chronologically using the extracted date parts so future loans do not inform past features — this prevents leakage that caused OOF AUC 0.9998 but LB score of only 0.58

train = train.sort_values(
    ['disbursement_date_year', 'disbursement_date_month', 'disbursement_date_day']
).reset_index(drop=True)

# shift(1) means each row only sees loans before it for that customer
train['cust_default_rate'] = (
    train.groupby('customer_id')['target']
    .transform(lambda x: x.shift(1).expanding().mean())
)
train['cust_total_loans'] = (
    train.groupby('customer_id').cumcount()   # 0 for first loan
)
train['cust_avg_amount'] = (
    train.groupby('customer_id')['Total_Amount']
    .transform(lambda x: x.shift(1).expanding().mean())
)
train['cust_avg_interest_rate'] = (
    train.groupby('customer_id')['interest_rate']
    .transform(lambda x: x.shift(1).expanding().mean())
)

# first loan for a customer has no prior history — fill with global stats
global_default_rate = train['target'].mean()
train['cust_default_rate']     = train['cust_default_rate'].fillna(global_default_rate)
train['cust_avg_amount']       = train['cust_avg_amount'].fillna(train['Total_Amount'].median())
train['cust_avg_interest_rate']= train['cust_avg_interest_rate'].fillna(train['interest_rate'].median())

# for test use the final snapshot of each customer at the end of training
final_cust_stats = train.groupby('customer_id').last().reset_index()

test = test.merge(
    final_cust_stats[['customer_id', 'cust_default_rate', 'cust_total_loans',
                       'cust_avg_amount', 'cust_avg_interest_rate']],
    on='customer_id', how='left'
)

# unknown customers in test get global defaults
test['cust_default_rate']      = test['cust_default_rate'].fillna(global_default_rate)
test['cust_total_loans']       = test['cust_total_loans'].fillna(0)
test['cust_avg_amount']        = test['cust_avg_amount'].fillna(train['Total_Amount'].median())
test['cust_avg_interest_rate'] = test['cust_avg_interest_rate'].fillna(train['interest_rate'].median())

print('customer history features added')
print('train cust_default_rate nulls:', train['cust_default_rate'].isnull().sum())
print('test  cust_default_rate nulls:', test['cust_default_rate'].isnull().sum())

customer history features added
train cust_default_rate nulls: 0
test  cust_default_rate nulls: 0


In [9]:
# loan_type has 19 types — rare types (<50 samples) are unreliable


freq = train['loan_type'].value_counts()
rare_types = freq[freq < 50].index.tolist()
print('Rare loan types (will be grouped):', rare_types)

for df in [train, test]:
    df['loan_type_grouped'] = df['loan_type'].apply(
        lambda x: 'Type_rare' if x in rare_types else x
    )

Rare loan types (will be grouped): ['Type_10', 'Type_15', 'Type_14', 'Type_20', 'Type_11', 'Type_17', 'Type_13', 'Type_21', 'Type_16', 'Type_12', 'Type_18', 'Type_19']


In [10]:
# loan_type has 19 types — rare types below 50 samples are unreliable


freq = train['loan_type'].value_counts()
rare_types = freq[freq < 50].index.tolist()
print('rare loan types grouped:', rare_types)

for df in [train, test]:
    df['loan_type_grouped'] = df['loan_type'].apply(
        lambda x: 'Type_rare' if x in rare_types else x
    )

rare loan types grouped: ['Type_10', 'Type_15', 'Type_14', 'Type_20', 'Type_11', 'Type_17', 'Type_13', 'Type_21', 'Type_16', 'Type_12', 'Type_18', 'Type_19']


In [11]:
# country_id: only Kenya — zero variance, drop
# ID, tbl_loan_id: row identifiers, not signals
# customer_id: used for aggregation above, raw ID not useful
# loan_type: replaced by loan_type_grouped
# New_versus_Repeat: replaced by is_repeat

DROP_COLS = ['country_id', 'ID', 'tbl_loan_id', 'customer_id',
             'loan_type', 'New_versus_Repeat']

train.drop(columns=[c for c in DROP_COLS if c in train.columns], inplace=True)
test.drop(columns=[c for c in DROP_COLS if c in test.columns],  inplace=True)

print('Train shape after FE + preprocessing:', train.shape)
print('Test  shape after FE + preprocessing:', test.shape)
print('\nFinal feature list:')
feature_cols = [c for c in train.columns if c != 'target']
for i, c in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {c}')

Train shape after FE + preprocessing: (68654, 41)
Test  shape after FE + preprocessing: (18594, 40)

Final feature list:
   1. lender_id
   2. Total_Amount
   3. Total_Amount_to_Repay
   4. duration
   5. Amount_Funded_By_Lender
   6. Lender_portion_Funded
   7. Lender_portion_to_be_repaid
   8. disbursement_date_month
   9. disbursement_date_dow
  10. disbursement_date_quarter
  11. disbursement_date_year
  12. disbursement_date_is_weekend
  13. disbursement_date_day
  14. due_date_month
  15. due_date_dow
  16. due_date_quarter
  17. due_date_year
  18. due_date_is_weekend
  19. due_date_day
  20. interest_charged
  21. interest_rate
  22. is_zero_interest
  23. is_negative_interest
  24. lender_exposure_ratio
  25. lender_return_ratio
  26. lender_funded_over1_flag
  27. lender_amount_per_day
  28. log_total_amount
  29. log_amount_to_repay
  30. amount_per_day
  31. repay_per_day
  32. is_7day_loan
  33. is_14day_loan
  34. is_30plus_loan
  35. log_duration
  36. cust_default_rate


In [12]:
# binary encode New_versus_Repeat if it still exists
for df in [train, test]:
    if 'New_versus_Repeat' in df.columns:
        df['is_repeat'] = (df['New_versus_Repeat'] == 'Repeat Loan').astype(int)
    elif 'is_repeat' not in df.columns:
        df['is_repeat'] = 0

# label encode loan_type_grouped and lender_id
for col in ['loan_type_grouped', 'lender_id']:
    if col in train.columns and col in test.columns:
        le = LabelEncoder()
        combined = pd.concat([train[col], test[col]], axis=0).astype(str)
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col]  = le.transform(test[col].astype(str))

In [13]:
# country_id is only Kenya — zero variance, drop it
# ID, tbl_loan_id, customer_id are identifiers not signals
# loan_type and New_versus_Repeat replaced by engineered versions above

DROP_COLS = ['country_id', 'ID', 'tbl_loan_id', 'customer_id',
             'loan_type', 'New_versus_Repeat']

train.drop(columns=[c for c in DROP_COLS if c in train.columns], inplace=True)
test.drop(columns=[c for c in DROP_COLS if c in test.columns],  inplace=True)

In [14]:
TARGET = 'target'
feature_cols = [c for c in train.columns if c != TARGET]

X      = train[feature_cols].copy()
y      = train[TARGET].astype(int).copy()
X_test = test[feature_cols].copy()

# align test columns to train in case any are missing after feature engineering
X_test = X_test.reindex(columns=X.columns)

# winsorise at 1st/99th — Total_Amount has a 20M outlier
for col in X.select_dtypes(include=[np.number]).columns:
    lo = X[col].quantile(0.01)
    hi = X[col].quantile(0.99)
    X[col]      = X[col].clip(lo, hi)
    X_test[col] = X_test[col].clip(lo, hi)

# impute any remaining nulls
imp    = SimpleImputer(strategy='median')
X      = pd.DataFrame(imp.fit_transform(X),      columns=feature_cols)
X_test = pd.DataFrame(imp.transform(X_test),     columns=feature_cols)

scale_pos_weight = (y == 0).sum() / (y == 1).sum()

print(f'X: {X.shape}  X_test: {X_test.shape}')
print(f'default rate (target=1): {y.mean()*100:.2f}%')
print(f'scale_pos_weight: {scale_pos_weight:.1f}')
print(f'missing in X: {X.isnull().sum().sum()}  missing in X_test: {X_test.isnull().sum().sum()}')

X: (68654, 41)  X_test: (18594, 41)
default rate (target=1): 1.83%
scale_pos_weight: 53.6
missing in X: 0  missing in X_test: 0


### 4. Train models


In [15]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
results = {}

def run_cv(model_name, model, X, y, skf, use_smote=False):
    oof_probs = np.zeros(len(y))
    fold_accs = []
    fold_aucs = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx].copy(), X.iloc[val_idx].copy()
        y_tr, y_val = y.iloc[tr_idx].copy(), y.iloc[val_idx].copy()

        if use_smote:
            sm = SMOTE(random_state=SEED, k_neighbors=5)
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

        model.fit(X_tr, y_tr)

        probs = model.predict_proba(X_val)[:, 1]
        oof_probs[val_idx] = probs
        fold_accs.append(accuracy_score(y_val, (probs >= 0.5).astype(int)))
        fold_aucs.append(roc_auc_score(y_val, probs))

        print(f'  [{model_name}] Fold {fold+1}/{N_FOLDS}  '
              f'Acc: {fold_accs[-1]:.4f}  AUC: {fold_aucs[-1]:.4f}')

    oof_acc = accuracy_score(y, (oof_probs >= 0.5).astype(int))
    oof_auc = roc_auc_score(y, oof_probs)

    print(f'\n  {model_name} | CV Acc: {np.mean(fold_accs):.4f} | '
          f'CV AUC: {np.mean(fold_aucs):.4f} | OOF AUC: {oof_auc:.4f}\n')

    results[model_name] = {
        'oof_probs': oof_probs,
        'cv_acc'   : np.mean(fold_accs),
        'cv_auc'   : np.mean(fold_aucs),
        'oof_acc'  : oof_acc,
        'oof_auc'  : oof_auc,
        'std_acc'  : np.std(fold_accs),
    }
    return oof_probs

In [16]:
# CatBoost handles categoricals natively and often complements XGBoost/LGBM adding it now so it contributes to the ensemble

print('CatBoost')

cb_params = dict(
    iterations        = 1000,
    learning_rate     = 0.05,
    depth             = 5,
    l2_leaf_reg       = 3.0,
    border_count      = 128,
    auto_class_weights= 'Balanced',
    eval_metric       = 'AUC',
    random_seed       = SEED,
    verbose           = 0
)

oof_cb        = np.zeros(len(y))
cb_accs       = []
cb_aucs       = []
cb_best_iters = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx].values, X.iloc[val_idx].values
    y_tr, y_val = y.iloc[tr_idx].values, y.iloc[val_idx].values

    m = CatBoostClassifier(**cb_params)
    m.fit(X_tr, y_tr,
          eval_set=(X_val, y_val),
          early_stopping_rounds=50,
          verbose=False)

    probs = m.predict_proba(X_val)[:, 1]
    oof_cb[val_idx] = probs
    cb_accs.append(accuracy_score(y_val, (probs >= 0.5).astype(int)))
    cb_aucs.append(roc_auc_score(y_val, probs))
    cb_best_iters.append(m.best_iteration_)
    print(f'  [CatBoost] Fold {fold+1}/{N_FOLDS}  '
          f'Acc: {cb_accs[-1]:.4f}  AUC: {cb_aucs[-1]:.4f}  '
          f'(best iter: {m.best_iteration_})')

cb_full_iters = int(np.mean(cb_best_iters) * (N_FOLDS / (N_FOLDS - 1)))

results['CatBoost'] = {
    'oof_probs'        : oof_cb,
    'cv_acc'           : np.mean(cb_accs),
    'cv_auc'           : np.mean(cb_aucs),
    'oof_acc'          : accuracy_score(y, (oof_cb >= 0.5).astype(int)),
    'oof_auc'          : roc_auc_score(y, oof_cb),
    'std_acc'          : np.std(cb_accs),
    'full_n_estimators': cb_full_iters
}
print(f'\n  CatBoost | CV Acc: {np.mean(cb_accs):.4f} | '
      f'CV AUC: {np.mean(cb_aucs):.4f} | OOF AUC: {results["CatBoost"]["oof_auc"]:.4f}\n')

CatBoost
  [CatBoost] Fold 1/5  Acc: 0.9908  AUC: 0.9952  (best iter: 408)
  [CatBoost] Fold 2/5  Acc: 0.9910  AUC: 0.9980  (best iter: 391)
  [CatBoost] Fold 3/5  Acc: 0.9873  AUC: 0.9978  (best iter: 295)
  [CatBoost] Fold 4/5  Acc: 0.9883  AUC: 0.9969  (best iter: 265)
  [CatBoost] Fold 5/5  Acc: 0.9844  AUC: 0.9981  (best iter: 228)

  CatBoost | CV Acc: 0.9884 | CV AUC: 0.9972 | OOF AUC: 0.9967



In [ ]:
# XGBoost is the current best LB model (0.6218)
# tuning it with Optuna Bayesian search across the parameters that matter most
# n_trials=60 balances search quality vs runtime 

print('tuning XGBoost with Optuna')

def xgb_objective(trial):
    params = dict(
        n_estimators          = trial.suggest_int('n_estimators', 200, 800),
        learning_rate         = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        max_depth             = trial.suggest_int('max_depth', 3, 7),
        subsample             = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree      = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha             = trial.suggest_float('reg_alpha', 0.01, 5.0, log=True),
        reg_lambda            = trial.suggest_float('reg_lambda', 0.01, 5.0, log=True),
        min_child_weight      = trial.suggest_int('min_child_weight', 1, 20),
        scale_pos_weight      = scale_pos_weight,
        eval_metric           = 'auc',
        random_state          = SEED,
        n_jobs                = -1,
        verbosity             = 0
    )
    fold_aucs = []
    for tr_idx, val_idx in skf.split(X, y):
        m = xgb.XGBClassifier(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              verbose=False)
        probs = m.predict_proba(X.iloc[val_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y.iloc[val_idx], probs))
    return np.mean(fold_aucs)

xgb_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
xgb_study.optimize(xgb_objective, n_trials=60, show_progress_bar=True)

print(f'best XGBoost AUC from tuning: {xgb_study.best_value:.4f}')
print(f'best params: {xgb_study.best_params}')

In [ ]:
# retrain tuned XGBoost on all 5 folds to get OOF probs for ensemble

print('retraining tuned XGBoost across folds')

best_xgb_params = {
    **xgb_study.best_params,
    'scale_pos_weight': scale_pos_weight,
    'eval_metric'     : 'auc',
    'random_state'    : SEED,
    'n_jobs'          : -1,
    'verbosity'       : 0
}

oof_xgb_tuned     = np.zeros(len(y))
xgb_tuned_accs    = []
xgb_tuned_aucs    = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    m = xgb.XGBClassifier(**best_xgb_params)
    m.fit(X_tr, y_tr,
          eval_set=[(X_val, y_val)],
          verbose=False)

    probs = m.predict_proba(X_val)[:, 1]
    oof_xgb_tuned[val_idx] = probs
    xgb_tuned_accs.append(accuracy_score(y_val, (probs >= 0.5).astype(int)))
    xgb_tuned_aucs.append(roc_auc_score(y_val, probs))
    print(f'  [XGBoost Tuned] Fold {fold+1}/{N_FOLDS}  '
          f'Acc: {xgb_tuned_accs[-1]:.4f}  AUC: {xgb_tuned_aucs[-1]:.4f}')

results['XGBoost_Tuned'] = {
    'oof_probs': oof_xgb_tuned,
    'cv_acc'   : np.mean(xgb_tuned_accs),
    'cv_auc'   : np.mean(xgb_tuned_aucs),
    'oof_acc'  : accuracy_score(y, (oof_xgb_tuned >= 0.5).astype(int)),
    'oof_auc'  : roc_auc_score(y, oof_xgb_tuned),
    'std_acc'  : np.std(xgb_tuned_accs),
}
print(f'\n  XGBoost Tuned | CV Acc: {np.mean(xgb_tuned_accs):.4f} | '
      f'CV AUC: {np.mean(xgb_tuned_aucs):.4f} | OOF AUC: {results["XGBoost_Tuned"]["oof_auc"]:.4f}')
if "XGBoost" in results:
    print(f'  baseline XGBoost OOF AUC was: {results["XGBoost"]["oof_auc"]:.4f}')
else:
    print('  baseline XGBoost OOF AUC not available in results.')

retraining tuned XGBoost across folds
  [XGBoost Tuned] Fold 1/5  Acc: 0.9918  AUC: 0.9962
  [XGBoost Tuned] Fold 1/5  Acc: 0.9918  AUC: 0.9962
  [XGBoost Tuned] Fold 2/5  Acc: 0.9910  AUC: 0.9979
  [XGBoost Tuned] Fold 2/5  Acc: 0.9910  AUC: 0.9979
  [XGBoost Tuned] Fold 3/5  Acc: 0.9895  AUC: 0.9979
  [XGBoost Tuned] Fold 3/5  Acc: 0.9895  AUC: 0.9979
  [XGBoost Tuned] Fold 4/5  Acc: 0.9932  AUC: 0.9970
  [XGBoost Tuned] Fold 4/5  Acc: 0.9932  AUC: 0.9970
  [XGBoost Tuned] Fold 5/5  Acc: 0.9883  AUC: 0.9982

  XGBoost Tuned | CV Acc: 0.9907 | CV AUC: 0.9974 | OOF AUC: 0.9973
  baseline XGBoost OOF AUC not available in results.
  [XGBoost Tuned] Fold 5/5  Acc: 0.9883  AUC: 0.9982

  XGBoost Tuned | CV Acc: 0.9907 | CV AUC: 0.9974 | OOF AUC: 0.9973
  baseline XGBoost OOF AUC not available in results.


In [23]:
# tune LightGBM as well — it had highest OOF AUC before tuning
# keeping num_leaves tightly bounded because high values caused the earlier leakage pattern

print('tuning LightGBM with Optuna')

def lgbm_objective(trial):
    params = dict(
        objective         = 'binary',
        metric            = 'auc',
        n_estimators      = trial.suggest_int('n_estimators', 200, 800),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        max_depth         = trial.suggest_int('max_depth', 3, 7),
        num_leaves        = trial.suggest_int('num_leaves', 10, 50),
        min_child_samples = trial.suggest_int('min_child_samples', 20, 100),
        subsample         = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 0.01, 5.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 0.01, 5.0, log=True),
        scale_pos_weight  = scale_pos_weight,
        random_state      = SEED,
        n_jobs            = -1,
        verbose           = -1
    )
    fold_aucs = []
    for tr_idx, val_idx in skf.split(X, y):
        m = lgb.LGBMClassifier(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              callbacks=[lgb.early_stopping(30, verbose=False),
                         lgb.log_evaluation(period=-1)])
        probs = m.predict_proba(X.iloc[val_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y.iloc[val_idx], probs))
    return np.mean(fold_aucs)

lgbm_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
lgbm_study.optimize(lgbm_objective, n_trials=60, show_progress_bar=True)

print(f'best LightGBM AUC from tuning: {lgbm_study.best_value:.4f}')
print(f'best params: {lgbm_study.best_params}')

tuning LightGBM with Optuna


  0%|          | 0/60 [00:00<?, ?it/s]

best LightGBM AUC from tuning: 0.9974
best params: {'n_estimators': 246, 'learning_rate': 0.06773430518632247, 'max_depth': 6, 'num_leaves': 39, 'min_child_samples': 83, 'subsample': 0.7065916059622793, 'colsample_bytree': 0.6800875130313317, 'reg_alpha': 0.019680497029015948, 'reg_lambda': 2.5295002349710276}


In [ ]:
# retrain tuned LightGBM across folds

print('retraining tuned LightGBM across folds')

best_lgbm_params = {
    **lgbm_study.best_params,
    'objective'       : 'binary',
    'metric'          : 'auc',
    'scale_pos_weight': scale_pos_weight,
    'random_state'    : SEED,
    'n_jobs'          : -1,
    'verbose'         : -1
}

oof_lgbm_tuned    = np.zeros(len(y))
lgbm_tuned_accs   = []
lgbm_tuned_aucs   = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    m = lgb.LGBMClassifier(**best_lgbm_params)
    m.fit(X_tr, y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(period=-1)])

    probs = m.predict_proba(X_val)[:, 1]
    oof_lgbm_tuned[val_idx] = probs
    lgbm_tuned_accs.append(accuracy_score(y_val, (probs >= 0.5).astype(int)))
    lgbm_tuned_aucs.append(roc_auc_score(y_val, probs))
    print(f'  [LightGBM Tuned] Fold {fold+1}/{N_FOLDS}  '
          f'Acc: {lgbm_tuned_accs[-1]:.4f}  AUC: {lgbm_tuned_aucs[-1]:.4f}')

results['LightGBM_Tuned'] = {
    'oof_probs': oof_lgbm_tuned,
    'cv_acc'   : np.mean(lgbm_tuned_accs),
    'cv_auc'   : np.mean(lgbm_tuned_aucs),
    'oof_acc'  : accuracy_score(y, (oof_lgbm_tuned >= 0.5).astype(int)),
    'oof_auc'  : roc_auc_score(y, oof_lgbm_tuned),
    'std_acc'  : np.std(lgbm_tuned_accs),
}
print(f'\n  LightGBM Tuned | CV Acc: {np.mean(lgbm_tuned_accs):.4f} | '
      f'CV AUC: {np.mean(lgbm_tuned_aucs):.4f} | OOF AUC: {results["LightGBM_Tuned"]["oof_auc"]:.4f}')

# avoid KeyError if baseline LightGBM wasn't run earlier
if "LightGBM" in results:
    print(f'  baseline LightGBM OOF AUC was: {results["LightGBM"]["oof_auc"]:.4f}')
else:
    print('  baseline LightGBM OOF AUC not available in results.')

retraining tuned LightGBM across folds
  [LightGBM Tuned] Fold 1/5  Acc: 0.9932  AUC: 0.9959
  [LightGBM Tuned] Fold 1/5  Acc: 0.9932  AUC: 0.9959
  [LightGBM Tuned] Fold 2/5  Acc: 0.9922  AUC: 0.9978
  [LightGBM Tuned] Fold 2/5  Acc: 0.9922  AUC: 0.9978
  [LightGBM Tuned] Fold 3/5  Acc: 0.9909  AUC: 0.9980
  [LightGBM Tuned] Fold 3/5  Acc: 0.9909  AUC: 0.9980
  [LightGBM Tuned] Fold 4/5  Acc: 0.9901  AUC: 0.9971
  [LightGBM Tuned] Fold 4/5  Acc: 0.9901  AUC: 0.9971
  [LightGBM Tuned] Fold 5/5  Acc: 0.9889  AUC: 0.9984

  LightGBM Tuned | CV Acc: 0.9911 | CV AUC: 0.9974 | OOF AUC: 0.9969
  baseline LightGBM OOF AUC not available in results.
  [LightGBM Tuned] Fold 5/5  Acc: 0.9889  AUC: 0.9984

  LightGBM Tuned | CV Acc: 0.9911 | CV AUC: 0.9974 | OOF AUC: 0.9969
  baseline LightGBM OOF AUC not available in results.


### 5. Select and finetune the best performing model


In [ ]:
# ensemble: weighted average of tuned XGB + tuned LGBM + CatBoost
# weights proportional to OOF AUC so better models get more say
# using these three because they are diverse (different algorithms, different skew handling)

ensemble_members = ['XGBoost_Tuned', 'LightGBM_Tuned', 'CatBoost']
auc_weights      = np.array([results[m]['oof_auc'] for m in ensemble_members])
auc_weights      = auc_weights / auc_weights.sum()

print('ensemble weights by OOF AUC:')
for m, w in zip(ensemble_members, auc_weights):
    print(f'  {m:<22} {w:.4f}')

oof_ensemble = sum(
    w * results[m]['oof_probs']
    for m, w in zip(ensemble_members, auc_weights)
)

ens_auc = roc_auc_score(y, oof_ensemble)
ens_acc = accuracy_score(y, (oof_ensemble >= 0.5).astype(int))

results['Ensemble'] = {
    'oof_probs': oof_ensemble,
    'cv_acc'   : ens_acc,
    'cv_auc'   : ens_auc,
    'oof_acc'  : ens_acc,
    'oof_auc'  : ens_auc,
    'std_acc'  : 0.0
}

print(f'\n  Ensemble OOF AUC: {ens_auc:.4f}  OOF Acc: {ens_acc:.4f}')
print(f'  tuned XGBoost alone: {results["XGBoost_Tuned"]["oof_auc"]:.4f}')
print(f'  gain from ensemble: {ens_auc - results["XGBoost_Tuned"]["oof_auc"]:+.4f}')

ensemble weights by OOF AUC:
  XGBoost_Tuned          0.3334
  LightGBM_Tuned         0.3333
  CatBoost               0.3333

  Ensemble OOF AUC: 0.9971  OOF Acc: 0.9904
  tuned XGBoost alone: 0.9973
  gain from ensemble: -0.0002


In [26]:
# full retrain on all training data then predict test
# each model uses its own best params, no early stopping needed (fixed n_estimators)

test_original = pd.read_csv('Test.csv')
test_ids      = test_original['ID']

# tuned XGBoost
print('training tuned XGBoost on full data...')
xgb_tuned_full = xgb.XGBClassifier(**best_xgb_params)
xgb_tuned_full.fit(X, y)
test_prob_xgb_tuned = xgb_tuned_full.predict_proba(X_test)[:, 1]
xgb_t_preds = (test_prob_xgb_tuned >= 0.5).astype(int)
pd.DataFrame({'ID': test_ids, 'target': xgb_t_preds}).to_csv('submission_xgb_tuned.csv', index=False)
print('saved submission_xgb_tuned.csv')

# tuned LightGBM
print('training tuned LightGBM on full data...')
lgbm_tuned_full = lgb.LGBMClassifier(**best_lgbm_params)
lgbm_tuned_full.fit(X, y)
test_prob_lgbm_tuned = lgbm_tuned_full.predict_proba(X_test)[:, 1]
lgbm_t_preds = (test_prob_lgbm_tuned >= 0.5).astype(int)
pd.DataFrame({'ID': test_ids, 'target': lgbm_t_preds}).to_csv('submission_lgbm_tuned.csv', index=False)
print('saved submission_lgbm_tuned.csv')

# CatBoost
print('training CatBoost on full data...')
cb_full_params = {**cb_params, 'iterations': results['CatBoost']['full_n_estimators']}
cb_full = CatBoostClassifier(**cb_full_params)
cb_full.fit(X.values, y.values, verbose=False)
test_prob_cb = cb_full.predict_proba(X_test.values)[:, 1]
cb_preds = (test_prob_cb >= 0.5).astype(int)
pd.DataFrame({'ID': test_ids, 'target': cb_preds}).to_csv('submission_cb.csv', index=False)
print('saved submission_cb.csv')

# ensemble: same weights as OOF ensemble above
test_prob_ensemble = (
    auc_weights[0] * test_prob_xgb_tuned +
    auc_weights[1] * test_prob_lgbm_tuned +
    auc_weights[2] * test_prob_cb
)
ens_preds = (test_prob_ensemble >= 0.5).astype(int)
pd.DataFrame({'ID': test_ids, 'target': ens_preds}).to_csv('submission_ensemble.csv', index=False)
print('saved submission_ensemble.csv')

print('\nall submissions saved')

training tuned XGBoost on full data...
saved submission_xgb_tuned.csv
training tuned LightGBM on full data...
saved submission_xgb_tuned.csv
training tuned LightGBM on full data...
saved submission_lgbm_tuned.csv
training CatBoost on full data...
saved submission_lgbm_tuned.csv
training CatBoost on full data...
saved submission_cb.csv
saved submission_ensemble.csv

all submissions saved
saved submission_cb.csv
saved submission_ensemble.csv

all submissions saved


### 6. Evaluate

In [ ]:
train_raw = pd.read_csv('Train.csv')
test_raw  = pd.read_csv('Test.csv')

train_customers = set(train_raw['customer_id'].unique())
test_customers  = set(test_raw['customer_id'].unique())

known   = test_customers & train_customers
unknown = test_customers - train_customers

print(f'test customers with training history : {len(known):,}  ({len(known)/len(test_customers)*100:.1f}%)')
print(f'test customers with NO history       : {len(unknown):,}  ({len(unknown)/len(test_customers)*100:.1f}%)')
print(f'\nthis is why cust_default_rate collapses on test')
print(f'unknown customers all get filled with global_default_rate={train_raw["target"].mean():.4f}')
print(f'the model learned nothing about how to score them')

test customers with training history : 4,225  (86.2%)
test customers with NO history       : 679  (13.8%)

this is why cust_default_rate collapses on test
unknown customers all get filled with global_default_rate=0.0183
the model learned nothing about how to score them


In [28]:
# re-read raw data to rebuild cleanly from this point
train = pd.read_csv('Train.csv')
test  = pd.read_csv('Test.csv')

# encode is_repeat BEFORE dropping New_versus_Repeat — it was lost before
for df in [train, test]:
    df['is_repeat'] = (df['New_versus_Repeat'] == 'Repeat Loan').astype(int)

# flag test customers who have no training history at all
# these rows had cust_default_rate filled with global mean — essentially blind
known_customers = set(train['customer_id'].unique())
test['is_new_to_training'] = (~test['customer_id'].isin(known_customers)).astype(int)
train['is_new_to_training'] = 0   # all train customers are known in training by definition

print(f'is_new_to_training in test: {test["is_new_to_training"].sum():,} '
      f'({test["is_new_to_training"].mean()*100:.1f}%)')

is_new_to_training in test: 3,790 (20.4%)


In [ ]:
for df in [train, test]:
    for col in ['disbursement_date', 'due_date']:
        dt = pd.to_datetime(df[col])
        df[f'{col}_month']      = dt.dt.month
        df[f'{col}_dow']        = dt.dt.dayofweek
        df[f'{col}_quarter']    = dt.dt.quarter
        df[f'{col}_year']       = dt.dt.year
        df[f'{col}_is_weekend'] = (dt.dt.dayofweek >= 5).astype(int)
        df[f'{col}_day']        = dt.dt.day
    df.drop(columns=['disbursement_date', 'due_date'], inplace=True)

for df in [train, test]:
    df['interest_charged']     = df['Total_Amount_to_Repay'] - df['Total_Amount']
    df['interest_rate']        = df['interest_charged'] / (df['Total_Amount'] + 1)
    df['is_zero_interest']     = (df['interest_charged'] == 0).astype(int)
    df['is_negative_interest'] = (df['interest_charged'] < 0).astype(int)

for df in [train, test]:
    df['lender_exposure_ratio']    = df['Amount_Funded_By_Lender'] / (df['Total_Amount'] + 1)
    df['lender_return_ratio']      = df['Lender_portion_to_be_repaid'] / (df['Amount_Funded_By_Lender'] + 1)
    df['lender_funded_over1_flag'] = (df['Lender_portion_Funded'] > 1).astype(int)
    df['lender_amount_per_day']    = df['Amount_Funded_By_Lender'] / (df['duration'] + 1)

for df in [train, test]:
    df['log_total_amount']    = np.log1p(df['Total_Amount'].clip(lower=0))
    df['log_amount_to_repay'] = np.log1p(df['Total_Amount_to_Repay'].clip(lower=0))
    df['amount_per_day']      = df['Total_Amount'] / (df['duration'] + 1)
    df['repay_per_day']       = df['Total_Amount_to_Repay'] / (df['duration'] + 1)

for df in [train, test]:
    df['is_7day_loan']   = (df['duration'] == 7).astype(int)
    df['is_14day_loan']  = (df['duration'] == 14).astype(int)
    df['is_30plus_loan'] = (df['duration'] >= 30).astype(int)
    df['log_duration']   = np.log1p(df['duration'])

# lender-level default rate — computed from train only, safe to use
lender_stats = train.groupby('lender_id')['target'].agg(
    lender_default_rate='mean',
    lender_loan_count='count'
).reset_index()
train = train.merge(lender_stats, on='lender_id', how='left')
test  = test.merge(lender_stats,  on='lender_id', how='left')
test['lender_default_rate'] = test['lender_default_rate'].fillna(train['target'].mean())
test['lender_loan_count']   = test['lender_loan_count'].fillna(0)

# loan_type default rate — computed from train only
lt_stats = train.groupby('loan_type')['target'].agg(
    loan_type_default_rate='mean'
).reset_index()
train = train.merge(lt_stats, on='loan_type', how='left')
test  = test.merge(lt_stats,  on='loan_type', how='left')
test['loan_type_default_rate'] = test['loan_type_default_rate'].fillna(train['target'].mean())

print('lender and loan_type default rates added')

lender and loan_type default rates added


In [30]:
global_default_rate = train['target'].mean()

train = train.sort_values(
    ['disbursement_date_year', 'disbursement_date_month', 'disbursement_date_day']
).reset_index(drop=True)

train['cust_default_rate'] = (
    train.groupby('customer_id')['target']
    .transform(lambda x: x.shift(1).expanding().mean())
)
train['cust_total_loans'] = train.groupby('customer_id').cumcount()

train['cust_avg_amount'] = (
    train.groupby('customer_id')['Total_Amount']
    .transform(lambda x: x.shift(1).expanding().mean())
)
train['cust_avg_interest_rate'] = (
    train.groupby('customer_id')['interest_rate']
    .transform(lambda x: x.shift(1).expanding().mean())
)

# new: is the customer on their very first loan — most uncertain
train['cust_is_first_loan'] = (train['cust_total_loans'] == 0).astype(int)

# new: volatility in loan amounts — erratic borrowing is a risk signal
train['cust_amount_std'] = (
    train.groupby('customer_id')['Total_Amount']
    .transform(lambda x: x.shift(1).expanding().std())
)

train['cust_default_rate']     = train['cust_default_rate'].fillna(global_default_rate)
train['cust_avg_amount']       = train['cust_avg_amount'].fillna(train['Total_Amount'].median())
train['cust_avg_interest_rate']= train['cust_avg_interest_rate'].fillna(train['interest_rate'].median())
train['cust_amount_std']       = train['cust_amount_std'].fillna(0)

final_cust = train.groupby('customer_id').last().reset_index()

test = test.merge(
    final_cust[['customer_id', 'cust_default_rate', 'cust_total_loans',
                'cust_avg_amount', 'cust_avg_interest_rate',
                'cust_is_first_loan', 'cust_amount_std']],
    on='customer_id', how='left'
)

# unknown test customers get global defaults — flag this explicitly
test['cust_default_rate']      = test['cust_default_rate'].fillna(global_default_rate)
test['cust_total_loans']       = test['cust_total_loans'].fillna(0)
test['cust_avg_amount']        = test['cust_avg_amount'].fillna(train['Total_Amount'].median())
test['cust_avg_interest_rate'] = test['cust_avg_interest_rate'].fillna(train['interest_rate'].median())
test['cust_is_first_loan']     = test['cust_is_first_loan'].fillna(1)   # unknown = treat as first
test['cust_amount_std']        = test['cust_amount_std'].fillna(0)

print('customer history features added')
print(f'train cust_default_rate nulls: {train["cust_default_rate"].isnull().sum()}')
print(f'test  cust_default_rate nulls: {test["cust_default_rate"].isnull().sum()}')

customer history features added
train cust_default_rate nulls: 0
test  cust_default_rate nulls: 0


In [ ]:
freq = train['loan_type'].value_counts()
rare_types = freq[freq < 50].index.tolist()
for df in [train, test]:
    df['loan_type_grouped'] = df['loan_type'].apply(
        lambda x: 'Type_rare' if x in rare_types else x
    )

for col in ['loan_type_grouped', 'lender_id']:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

DROP_COLS = ['country_id', 'ID', 'tbl_loan_id', 'customer_id',
             'loan_type', 'New_versus_Repeat']
train.drop(columns=[c for c in DROP_COLS if c in train.columns], inplace=True)
test.drop(columns=[c for c in DROP_COLS if c in test.columns],  inplace=True)

TARGET       = 'target'
feature_cols = [c for c in train.columns if c != TARGET]
X      = train[feature_cols].copy()
y      = train[TARGET].astype(int).copy()
X_test = test[feature_cols].copy()
X_test = X_test.reindex(columns=X.columns)

for col in X.select_dtypes(include=[np.number]).columns:
    lo = X[col].quantile(0.01)
    hi = X[col].quantile(0.99)
    X[col]      = X[col].clip(lo, hi)
    X_test[col] = X_test[col].clip(lo, hi)

imp    = SimpleImputer(strategy='median')
X      = pd.DataFrame(imp.fit_transform(X),  columns=feature_cols)
X_test = pd.DataFrame(imp.transform(X_test), columns=feature_cols)

scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print(f'X: {X.shape}  X_test: {X_test.shape}')
print(f'features: {X.shape[1]}')
print(f'new features added: is_repeat, is_new_to_training, cust_is_first_loan, '
      f'cust_amount_std, lender_default_rate, lender_loan_count, loan_type_default_rate')

X: (68654, 47)  X_test: (18594, 47)
features: 47
new features added: is_repeat, is_new_to_training, cust_is_first_loan, cust_amount_std, lender_default_rate, lender_loan_count, loan_type_default_rate


In [ ]:
# stacking uses OOF predictions from base models as features for a meta-learner
# the meta-learner learns WHICH model to trust for which kind of loan
# this is stronger than fixed-weight averaging

from sklearn.linear_model import LogisticRegression

# collect OOF probs from all tuned models
stacking_members = ['XGBoost_Tuned', 'LightGBM_Tuned', 'CatBoost']
meta_X_train = np.column_stack([results[m]['oof_probs'] for m in stacking_members])
meta_X_train = pd.DataFrame(meta_X_train, columns=stacking_members)

print('meta-feature correlations (high = models too similar):')
print(pd.DataFrame(meta_X_train).corr().round(3))

# train meta-learner using the same CV folds to avoid leakage
oof_stack  = np.zeros(len(y))
stack_aucs = []

for tr_idx, val_idx in skf.split(meta_X_train, y):
    meta_model = LogisticRegression(C=0.01, max_iter=1000, random_state=SEED)
    meta_model.fit(meta_X_train.iloc[tr_idx], y.iloc[tr_idx])
    probs = meta_model.predict_proba(meta_X_train.iloc[val_idx])[:, 1]
    oof_stack[val_idx] = probs
    stack_aucs.append(roc_auc_score(y.iloc[val_idx], probs))

stack_auc = roc_auc_score(y, oof_stack)
print(f'\nstacking OOF AUC : {stack_auc:.4f}')
print(f'weighted avg AUC : {results["Ensemble"]["oof_auc"]:.4f}')
print(f'gain from stacking: {stack_auc - results["Ensemble"]["oof_auc"]:+.4f}')

results['Stacking'] = {
    'oof_probs': oof_stack,
    'oof_auc'  : stack_auc,
    'cv_auc'   : np.mean(stack_aucs),
    'cv_acc'   : accuracy_score(y, (oof_stack >= 0.5).astype(int)),
    'oof_acc'  : accuracy_score(y, (oof_stack >= 0.5).astype(int)),
    'std_acc'  : np.std(stack_aucs)
}

meta-feature correlations (high = models too similar):
                XGBoost_Tuned  LightGBM_Tuned  CatBoost
XGBoost_Tuned           1.000           0.979     0.969
LightGBM_Tuned          0.979           1.000     0.965
CatBoost                0.969           0.965     1.000

stacking OOF AUC : 0.9961
weighted avg AUC : 0.9971
gain from stacking: -0.0010

stacking OOF AUC : 0.9961
weighted avg AUC : 0.9971
gain from stacking: -0.0010


In [33]:
# rank averaging converts each model's probs to ranks then averages
# this removes calibration differences between models
# often outperforms both weighted average and stacking when models are similar

from scipy.stats import rankdata

rank_avg = np.zeros(len(y))
for m in stacking_members:
    rank_avg += rankdata(results[m]['oof_probs'])
rank_avg /= len(stacking_members)
rank_avg_norm = rank_avg / rank_avg.max()

rank_auc = roc_auc_score(y, rank_avg_norm)
print(f'rank average OOF AUC : {rank_auc:.4f}')
print(f'weighted avg OOF AUC : {results["Ensemble"]["oof_auc"]:.4f}')
print(f'stacking OOF AUC     : {results["Stacking"]["oof_auc"]:.4f}')

results['RankAvg'] = {
    'oof_probs': rank_avg_norm,
    'oof_auc'  : rank_auc,
    'cv_auc'   : rank_auc,
    'cv_acc'   : accuracy_score(y, (rank_avg_norm >= 0.5).astype(int)),
    'oof_acc'  : accuracy_score(y, (rank_avg_norm >= 0.5).astype(int)),
    'std_acc'  : 0.0
}

rank average OOF AUC : 0.9973
weighted avg OOF AUC : 0.9971
stacking OOF AUC     : 0.9961


In [ ]:
test_original = pd.read_csv('Test.csv')
test_ids = test_original['ID']

# train final models on all data
xgb_final = xgb.XGBClassifier(**best_xgb_params)
xgb_final.fit(X, y)
tp_xgb = xgb_final.predict_proba(X_test)[:, 1]

lgbm_final = lgb.LGBMClassifier(**best_lgbm_params)
lgbm_final.fit(X, y)
tp_lgbm = lgbm_final.predict_proba(X_test)[:, 1]

cb_final = CatBoostClassifier(**{**cb_params,
                                  'iterations': results['CatBoost']['full_n_estimators']})
cb_final.fit(X.values, y.values, verbose=False)
tp_cb = cb_final.predict_proba(X_test.values)[:, 1]

# train meta-learner on full training OOF then apply to test
meta_model_full = LogisticRegression(C=0.01, max_iter=1000, random_state=SEED)
meta_model_full.fit(meta_X_train, y)
meta_X_test = pd.DataFrame(
    np.column_stack([tp_xgb, tp_lgbm, tp_cb]),
    columns=stacking_members
)
tp_stack = meta_model_full.predict_proba(meta_X_test)[:, 1]

# rank average on test
tp_rank = (rankdata(tp_xgb) + rankdata(tp_lgbm) + rankdata(tp_cb)) / 3
tp_rank = tp_rank / tp_rank.max()

# save all
for name, probs in [('xgb_tuned', tp_xgb), ('lgbm_tuned', tp_lgbm),
                    ('catboost', tp_cb), ('stacking', tp_stack),
                    ('rank_avg', tp_rank)]:
    preds = (probs >= 0.5).astype(int)
    pd.DataFrame({'ID': test_ids, 'target': preds}).to_csv(
        f'submission_{name}_v2.csv', index=False)
    print(f'saved submission_{name}_v2.csv  '
          f'(predicted default rate: {preds.mean()*100:.2f}%)')

saved submission_xgb_tuned_v2.csv  (predicted default rate: 4.74%)
saved submission_lgbm_tuned_v2.csv  (predicted default rate: 4.61%)
saved submission_catboost_v2.csv  (predicted default rate: 4.78%)
saved submission_stacking_v2.csv  (predicted default rate: 4.04%)
saved submission_rank_avg_v2.csv  (predicted default rate: 47.47%)


In [35]:
print(f"{'Model':<22} {'OOF AUC':>9} {'LB Score':>10}")
lb = {
    'XGBoost_Tuned' : None,
    'LightGBM_Tuned': None,
    'CatBoost'      : None,
    'Ensemble'      : 0.615776081,
    'Stacking'      : None,
    'RankAvg'       : None,
}
for name, score in lb.items():
    oof = results[name]['oof_auc'] if name in results else 0.0
    lb_str = f'{score:.6f}' if score is not None else 'pending'
    print(f'{name:<22} {oof:>9.4f} {lb_str:>10}')

print('\nsubmit in this order — stop if one beats 0.621:')
print('  1. submission_stacking_v2.csv    (meta-learner, should generalise best)')
print('  2. submission_rank_avg_v2.csv    (robust to calibration differences)')
print('  3. submission_xgb_tuned_v2.csv   (best single model historically)')

print('\nif OOF AUC is still ~0.997 after rebuilding features,')
print('run this sanity check to confirm cust_default_rate is the dominant feature:')

xgb_imp = pd.Series(xgb_final.feature_importances_, index=X.columns)
print(xgb_imp.sort_values(ascending=False).head(10).to_string())
print('\nif cust_default_rate is top by a large margin, try one run without it')

Model                    OOF AUC   LB Score
XGBoost_Tuned             0.9973    pending
LightGBM_Tuned            0.9969    pending
CatBoost                  0.9967    pending
Ensemble                  0.9971   0.615776
Stacking                  0.9961    pending
RankAvg                   0.9973    pending

submit in this order — stop if one beats 0.621:
  1. submission_stacking_v2.csv    (meta-learner, should generalise best)
  2. submission_rank_avg_v2.csv    (robust to calibration differences)
  3. submission_xgb_tuned_v2.csv   (best single model historically)

if OOF AUC is still ~0.997 after rebuilding features,
run this sanity check to confirm cust_default_rate is the dominant feature:
lender_return_ratio      0.463033
loan_type_grouped        0.142104
interest_rate            0.087864
interest_charged         0.040226
cust_default_rate        0.025974
duration                 0.024519
due_date_year            0.023931
due_date_month           0.017275
disbursement_date_day    0.